# VR Cybersickness — Dual-Task Benchmark

**Task 1**: Minute-level score regression + binary classification (LOSO-CV)  
**Task 2**: Session-level SSQ Total Score regression (LOSO-CV)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('.').resolve()))

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED   = 42
np.random.seed(SEED); torch.manual_seed(SEED)
print(f'Device: {DEVICE}')

## Task 1 — Minute-Level Prediction

In [ ]:
from tqdm.auto import tqdm
from src.utils import loso_split, fit_transform_X, eval_regression
from features.sequence_builder import FEAT_COLS, make_windows
from models.baselines import train_xgb, train_lr
from models.sequence_models import GRUPredictor
from models.fusion_network import MultimodalFusion

df = pd.read_excel('data/raw/dataset_sample.xlsx')
df['sick'] = (df['score'] >= 4.0).astype(int)
PARTICIPANTS = sorted(df['participant'].unique())
print(f'Participants: {PARTICIPANTS}, rows: {len(df)}')

In [ ]:
# XGBoost baseline
y_true_xgb, y_pred_xgb = [], []
for p in tqdm(PARTICIPANTS, desc='XGBoost'):
    tr, te = loso_split(df, p)
    X_tr, X_te = fit_transform_X(tr[FEAT_COLS].values, te[FEAT_COLS].values)
    m = train_xgb(X_tr, tr['score'].values, seed=SEED)
    y_pred_xgb.extend(m.predict(X_te)); y_true_xgb.extend(te['score'].values)

xgb_metrics = eval_regression(np.array(y_true_xgb), np.array(y_pred_xgb))
print('XGBoost:', xgb_metrics)

In [ ]:
# Linear Regression baseline
y_true_lr, y_pred_lr = [], []
for p in tqdm(PARTICIPANTS, desc='LR'):
    tr, te = loso_split(df, p)
    X_tr, X_te = fit_transform_X(tr[FEAT_COLS].values, te[FEAT_COLS].values)
    m = train_lr(X_tr, tr['score'].values)
    y_pred_lr.extend(m.predict(X_te)); y_true_lr.extend(te['score'].values)

lr_metrics = eval_regression(np.array(y_true_lr), np.array(y_pred_lr))
print('LR:', lr_metrics)

In [ ]:
# Task 1 summary table
task1_df = pd.DataFrame({
    'Model': ['Linear Regression', 'XGBoost'],
    'RMSE':  [lr_metrics['RMSE'],  xgb_metrics['RMSE']],
    'MAE':   [lr_metrics['MAE'],   xgb_metrics['MAE']],
    'r':     [lr_metrics['r'],     xgb_metrics['r']],
})
print('\n[Task 1: Minute-Level Regression]')
print(task1_df.to_string(index=False))

## Task 2 — Session-Level SSQ Prediction

In [ ]:
import pickle

X_seq    = np.load('data/raw/X_sequences.npy')
y_ssq    = np.load('data/raw/y_labels.npy')
lengths  = np.load('data/raw/seq_lengths.npy')
with open('data/raw/session_info.pkl', 'rb') as f:
    session_info = pickle.load(f)

PARTICIPANTS_T2 = sorted({s['participant'] for s in session_info})
print(f'Sessions: {len(X_seq)}, SSQ range: {y_ssq.min():.1f}–{y_ssq.max():.1f}')

In [ ]:
# Aggregated XGBoost
y_true_axgb, y_pred_axgb = [], []
for p in tqdm(PARTICIPANTS_T2, desc='Agg-XGB'):
    tr_idx = [i for i, s in enumerate(session_info) if s['participant'] != p]
    te_idx = [i for i, s in enumerate(session_info) if s['participant'] == p]
    X_tr = np.hstack([X_seq[tr_idx].mean(1), X_seq[tr_idx].std(1)])
    X_te = np.hstack([X_seq[te_idx].mean(1), X_seq[te_idx].std(1)])
    X_tr, X_te = fit_transform_X(X_tr, X_te)
    m = train_xgb(X_tr, y_ssq[tr_idx], seed=SEED)
    y_pred_axgb.extend(m.predict(X_te)); y_true_axgb.extend(y_ssq[te_idx])

axgb_metrics = eval_regression(np.array(y_true_axgb), np.array(y_pred_axgb))
print('Agg-XGBoost:', axgb_metrics)

In [ ]:
# Task 2 summary table
task2_df = pd.DataFrame({
    'Model': ['Agg-XGBoost'],
    'RMSE':  [axgb_metrics['RMSE']],
    'MAE':   [axgb_metrics['MAE']],
    'r':     [axgb_metrics['r']],
})
print('\n[Task 2: Session-Level SSQ Regression]')
print(task2_df.to_string(index=False))